In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, broadcast, concat, floor, when
import time

In [2]:
spark = (
    SparkSession.builder
    .appName("SparkPartitionBenchmark")
    .getOrCreate()
)

In [3]:
# Dimension table: 1M rows
dim = (
    spark.range(0, 1_000_000)
    .withColumnRenamed("id", "user_id")
)

# Fact table: 50M rows
fact = (
    spark.range(0, 50_000_000)
    .withColumn("user_id", col("id") % 1_000_000)
    .withColumn("value", rand())
)

In [4]:
# Fact table: 50M rows
fact = (
    spark.range(0, 50_000_000)
    .withColumn("user_id", col("id") % 1_000_000)
    .withColumn("value", rand())
)

In [5]:
def benchmark(name, fn):
    start = time.time()
    fn()
    duration = time.time() - start
    print(f"{name}: {duration:.2f} sec")
    return duration


In [6]:
# Narrow transformation (no shuffle)
benchmark(
    "Narrow Transformation",
    lambda: fact.filter(col("user_id") > 100).count()
)

# Wide transformation (shuffle)
spark.conf.set("spark.sql.shuffle.partitions", 200)

benchmark(
    "Wide Transformation (groupBy)",
    lambda: fact.groupBy("user_id").count().count()
)


Narrow Transformation: 1.48 sec
Wide Transformation (groupBy): 3.88 sec


3.877795934677124

In [7]:
cores = spark.sparkContext.defaultParallelism
print("Detected cores:", cores)

def run_groupby(partitions):
    spark.conf.set("spark.sql.shuffle.partitions", partitions)
    return benchmark(
        f"groupBy with {partitions} partitions",
        lambda: fact.groupBy("user_id").count().count()
    )

run_groupby(cores)           # too few
run_groupby(cores * 2)       # good
run_groupby(cores * 4)       # good
run_groupby(cores * 10)      # too many

Detected cores: 6
groupBy with 6 partitions: 3.87 sec
groupBy with 12 partitions: 2.39 sec
groupBy with 24 partitions: 2.51 sec
groupBy with 60 partitions: 2.94 sec


2.940333604812622

In [8]:
# Shuffle join
benchmark(
    "Shuffle Join",
    lambda: fact.join(dim, "user_id").count()
)

# Broadcast join
benchmark(
    "Broadcast Join",
    lambda: fact.join(broadcast(dim), "user_id").count()
)


Shuffle Join: 0.95 sec
Broadcast Join: 0.55 sec


0.5500814914703369

In [20]:
#Make sure data causes spill from main memory
# Dimension table: 1M rows
dim = (
    spark.range(0, 10_000_000)
    .withColumnRenamed("id", "user_id")
)

# Fact table: 50M rows
fact = (
    spark.range(0, 500_000_000)
    .withColumn("user_id", col("id") % 10_000_000)
    .withColumn("value", rand())
)

# Create skew (hot key)
skewed = fact.withColumn(
    "skewed_key",
    when(col("user_id") == 1, 1)
    .otherwise(col("user_id"))
)

benchmark(
    "Skewed Join (no mitigation)",
    lambda: skewed.join(dim, skewed.skewed_key == dim.user_id).count()
)

# Salting
salted = skewed.withColumn(
    "salt",
    floor(rand() * 10)
).withColumn(
    "salted_key",
    concat(col("skewed_key"), col("salt"))
)

dim_salted = dim.withColumn(
    "salt",
    floor(rand() * 10)
).withColumn(
    "salted_key",
    concat(col("user_id"), col("salt"))
)

benchmark(
    "Skewed Join (salting)",
    lambda: salted.join(dim_salted, "salted_key").count()
)


Skewed Join (no mitigation): 113.99 sec
Skewed Join (salting): 257.43 sec


257.4303984642029

In [19]:
agg = fact.groupBy("user_id").count()

benchmark("Without Cache", lambda: agg.count())

agg.cache()
agg.count()  # materialize

benchmark("With Cache", lambda: agg.count())


Without Cache: 2.75 sec
With Cache: 0.10 sec


0.09654688835144043